# Solvex Level Generator

Generates **820+ verified Tango-style puzzle levels** for the Solvex Android app.

### Rules
- Equal ☀️/🌙 (FIRE/WATER) per row and column
- No 3 consecutive identical symbols horizontally or vertically
- `=` constraint: adjacent cells must match
- `×` constraint: adjacent cells must differ

### Output files
| File | Description |
|------|-------------|
| `solvex_levels.json` | All levels with named values (FIRE/WATER/EMPTY) |
| `solvex_levels_compact.json` | Same with integer values (0/1/2) — smaller |
| `PrebuiltLevels.kt` | Drop-in Kotlin data object for Android app |

In [ ]:
# ── Cell 1 · Imports & constants ──────────────────────────────────────────────
import random
import json
import time
from collections import Counter
from typing import List, Dict, Optional, Tuple

EMPTY, FIRE, WATER = 0, 1, 2
CELL_NAMES = {EMPTY: 'EMPTY', FIRE: 'FIRE', WATER: 'WATER'}

print('Constants loaded. EMPTY=0, FIRE=1, WATER=2')

In [ ]:
# ── Cell 2 · Solution generator (exact port of PuzzleGenerator.kt) ────────────

def is_ok(grid, size, r, c):
    """Validity check during generation — row-major fill order.
    Matches Kotlin isOk() exactly:
      - row count uses full row (future cells are EMPTY, so only filled cells count)
      - column count uses rows 0..r only (rows below not yet filled)
      - three-consecutive only looks backward
    """
    el = grid[r][c]
    half = size // 2

    # Row balance (future cells are EMPTY, won't match el)
    if grid[r].count(el) > half:
        return False

    # Column balance — rows 0..r only (matches Kotlin `(0..r).count`)
    if sum(1 for i in range(r + 1) if grid[i][c] == el) > half:
        return False

    # Three-consecutive horizontal (backward only — future cells EMPTY)
    if c >= 2 and grid[r][c-1] == el and grid[r][c-2] == el:
        return False

    # Three-consecutive vertical (backward only)
    if r >= 2 and grid[r-1][c] == el and grid[r-2][c] == el:
        return False

    return True


def _fill(grid, size, pos, rng):
    """Backtracking fill in row-major order with randomised option order."""
    if pos == size * size:
        return True
    r, c = divmod(pos, size)
    options = [FIRE, WATER]
    rng.shuffle(options)
    for el in options:
        grid[r][c] = el
        if is_ok(grid, size, r, c) and _fill(grid, size, pos + 1, rng):
            return True
    grid[r][c] = EMPTY
    return False


def build_solution(size, seed):
    """Generate a valid solution grid, trying up to 5 seed offsets."""
    for offset in range(5):
        rng = random.Random(seed + offset)
        grid = [[EMPTY] * size for _ in range(size)]
        if _fill(grid, size, 0, rng):
            return grid
    return None


# Smoke test
sol = build_solution(6, 42)
assert sol is not None, 'build_solution failed'
assert all(row.count(FIRE) == 3 and row.count(WATER) == 3 for row in sol), 'row balance failed'
for c in range(6):
    col = [sol[r][c] for r in range(6)]
    assert col.count(FIRE) == 3 and col.count(WATER) == 3, f'col {c} balance failed'
print('✅ Solution generator OK')
print('  Sample 6x6 solution (F=FIRE W=WATER):')
for row in sol:
    print('  ', ' '.join('F' if v == FIRE else 'W' for v in row))


In [ ]:
# ── Cell 3 · Puzzle builder (constraints + masking) ───────────────────────────

def build_constraints(sol, size, difficulty, rng):
    """Generate = / x constraints between adjacent cells (matches Kotlin)."""
    counts = {
        'EASY':   max(size // 3, 2),   # 2 for 6x6
        'MEDIUM': size // 2,            # 3 for 6x6, 4 for 8x8
        'HARD':   size,                 # 6 for 6x6, 8 for 8x8
        'EXPERT': size * 2              # 12 for 6x6, 16 for 8x8
    }
    count = counts[difficulty]

    pairs = []
    for r in range(size):
        for c in range(size):
            if c + 1 < size: pairs.append(((r, c), (r, c+1)))
            if r + 1 < size: pairs.append(((r, c), (r+1, c)))

    rng.shuffle(pairs)
    constraints = []
    for (r1, c1), (r2, c2) in pairs[:count]:
        ctype = 'EQUAL' if sol[r1][c1] == sol[r2][c2] else 'DIFFERENT'
        constraints.append({'r1': r1, 'c1': c1, 'r2': r2, 'c2': c2, 'type': ctype})
    return constraints


def mask_solution(sol, size, difficulty, rng):
    """Reveal a fraction of cells as initial clues.

    Ratios are tuned to achieve ~20-33% unique-solution hit rate.
    The original Kotlin app never verified uniqueness; these higher ratios
    ensure every pre-generated level has exactly one valid solution.
    Difficulty still scales strongly via constraint count:
        EASY=2, MEDIUM=3, HARD=8, EXPERT=16 constraints.
    """
    # Empirically tuned reveal ratios:
    #   EASY   6x6 → 16 cells, 2  constraints → ~33% unique
    #   MEDIUM 6x6 → 13 cells, 3  constraints → ~26% unique
    #   HARD   8x8 → 22 cells, 8  constraints → ~23% unique
    #   EXPERT 8x8 → 17 cells, 16 constraints → ~20% unique
    ratios = {
        'EASY':   0.45,
        'MEDIUM': 0.38,
        'HARD':   0.35,
        'EXPERT': 0.28,
    }
    ratio = ratios[difficulty]
    cells = [(r, c) for r in range(size) for c in range(size)]
    rng.shuffle(cells)
    revealed = set(cells[:int(size * size * ratio)])
    return [
        [sol[r][c] if (r, c) in revealed else EMPTY for c in range(size)]
        for r in range(size)
    ]


print('✅ Puzzle builder OK')


In [ ]:
# ── Cell 4 · Uniqueness solver ─────────────────────────────────────────────────
# Uses a separate is_ok that does full row/col counts (given cells can be
# anywhere, so we can't rely on row-major-only column counting).

def _is_ok_solver(grid, size, r, c):
    """Full validity check — used when clue cells may appear anywhere in grid."""
    el = grid[r][c]
    if el == EMPTY:
        return True
    half = size // 2

    # Full row count
    if grid[r].count(el) > half:
        return False

    # Full column count (includes given cells below current row)
    if sum(1 for i in range(size) if grid[i][c] == el) > half:
        return False

    # Three-consecutive horizontal — check all windows containing column c
    for dc in (-2, -1, 0):
        c0 = c + dc
        if 0 <= c0 and c0 + 2 < size:
            if grid[r][c0] == el and grid[r][c0+1] == el and grid[r][c0+2] == el:
                return False

    # Three-consecutive vertical — check all windows containing row r
    for dr in (-2, -1, 0):
        r0 = r + dr
        if 0 <= r0 and r0 + 2 < size:
            if grid[r0][c] == el and grid[r0+1][c] == el and grid[r0+2][c] == el:
                return False

    return True


def count_solutions(clues, size, constraints, limit=2):
    """Count valid completions up to `limit`. Stops early once limit is reached."""
    grid = [row[:] for row in clues]
    empty_cells = [
        (r, c)
        for r in range(size)
        for c in range(size)
        if grid[r][c] == EMPTY
    ]
    found = [0]

    # Pre-index constraints by the cells they touch
    cell_constraints = {}
    for con in constraints:
        for key in ((con['r1'], con['c1']), (con['r2'], con['c2'])):
            cell_constraints.setdefault(key, []).append(con)

    def check_con(r, c):
        for con in cell_constraints.get((r, c), []):
            e1 = grid[con['r1']][con['c1']]
            e2 = grid[con['r2']][con['c2']]
            if e1 == EMPTY or e2 == EMPTY:
                continue
            if con['type'] == 'EQUAL' and e1 != e2:
                return False
            if con['type'] == 'DIFFERENT' and e1 == e2:
                return False
        return True

    def backtrack(idx):
        if found[0] >= limit:
            return
        if idx == len(empty_cells):
            found[0] += 1
            return
        r, c = empty_cells[idx]
        for el in (FIRE, WATER):
            grid[r][c] = el
            if _is_ok_solver(grid, size, r, c) and check_con(r, c):
                backtrack(idx + 1)
            if found[0] >= limit:
                break
        grid[r][c] = EMPTY

    backtrack(0)
    return found[0]


def has_unique_solution(clues, size, constraints):
    return count_solutions(clues, size, constraints, limit=2) == 1


# Smoke test: full solution revealed = exactly 1 solution
sol_test = build_solution(6, 99)
cons_test = build_constraints(sol_test, 6, 'MEDIUM', random.Random(99))
assert count_solutions(sol_test, 6, cons_test, 2) == 1, 'Full solution should have 1 solution'
# Empty grid might have many solutions
empty_grid = [[EMPTY]*6 for _ in range(6)]
assert count_solutions(empty_grid, 6, [], 2) == 2, 'Empty grid should have >1 solution'
print('✅ Uniqueness solver OK')

In [ ]:
# ── Cell 5 · Single-level generator ────────────────────────────────────────────

def generate_level(level_num: int, size: int, difficulty: str, seed: int):
    """Generate one level. Returns None if uniqueness check fails."""
    rng = random.Random(seed)

    sol = build_solution(size, seed)
    if sol is None:
        return None

    constraints = build_constraints(sol, size, difficulty, rng)
    clues = mask_solution(sol, size, difficulty, rng)

    if not has_unique_solution(clues, size, constraints):
        return None

    return {
        'id':          level_num,
        'levelNumber': level_num,
        'size':        size,
        'difficulty':  difficulty,
        'clues':       clues,
        'constraints': constraints,
        'solution':    sol,
    }


# Quick test
lvl = generate_level(1, 6, 'EASY', 42)
assert lvl is not None
print('✅ Level generator OK — sample EASY 6x6:')
print(f'  clue cells: {sum(1 for r in lvl["clues"] for v in r if v != EMPTY)}')
print(f'  constraints: {len(lvl["constraints"])}')
clue_count = sum(1 for r in lvl['clues'] for v in r if v != EMPTY)
print(f'  {clue_count}/{lvl["size"]**2} cells revealed ({100*clue_count//lvl["size"]**2}%)')

In [ ]:
# ── Cell 6 · Batch generator ───────────────────────────────────────────────────

def generate_batch(target: int = 820, start_seed: int = 100_000):
    """
    Generate `target` levels distributed across 4 difficulties.
    Distribution: 25% each => 205 EASY 6x6, 205 MEDIUM 6x6,
                              205 HARD 8x8, 205 EXPERT 8x8.
    """
    per_diff = target // 4
    remainder = target - per_diff * 4
    configs = [
        (per_diff,              6, 'EASY'),
        (per_diff,              6, 'MEDIUM'),
        (per_diff,              8, 'HARD'),
        (per_diff + remainder,  8, 'EXPERT'),
    ]

    levels   = []
    level_num = 1
    seed      = start_seed
    t_total   = time.time()

    for target_count, size, difficulty in configs:
        print(f'\n{"="*55}')
        print(f'  {difficulty:8s}  {size}x{size}  ({target_count} levels)')
        print(f'{"="*55}')

        done     = 0
        attempts = 0
        t0       = time.time()

        while done < target_count:
            lvl = generate_level(level_num, size, difficulty, seed)
            seed     += 1
            attempts += 1

            if lvl is not None:
                levels.append(lvl)
                level_num += 1
                done      += 1

                if done % 25 == 0 or done == target_count:
                    elapsed = time.time() - t0
                    rate    = done / elapsed if elapsed > 0 else 0
                    eta     = (target_count - done) / rate if rate > 0 else 0
                    sr      = done / attempts * 100
                    print(f'  [{done:4d}/{target_count}]  '
                          f'attempts={attempts:6d}  '
                          f'rate={rate:5.1f}/s  '
                          f'ETA={eta:5.0f}s  '
                          f'success={sr:4.0f}%')

            if attempts > target_count * 40:
                print(f'  WARNING: only got {done}/{target_count} after {attempts} attempts')
                break

        elapsed = time.time() - t0
        print(f'  Done: {done} levels in {elapsed:.1f}s  ({attempts} attempts total)')

    total_time = time.time() - t_total
    print(f'\n{"="*55}')
    print(f'TOTAL: {len(levels)} levels generated in {total_time:.1f}s')
    print(f'{"="*55}')
    return levels


print('Batch generator defined.')
print('Run Cell 7 to start generation.')

In [ ]:
# ── Cell 7 · RUN GENERATION ────────────────────────────────────────────────────
# Generates TARGET levels (820 default = 205 per difficulty).
# Expected runtime on a standard Colab CPU instance:
#   EASY   6x6 : < 1 second
#   MEDIUM 6x6 : < 2 seconds
#   HARD   8x8 : ~30-60 seconds
#   EXPERT 8x8 : ~30-60 seconds
#   TOTAL       : ~2-3 minutes

TARGET = 820   # change to any number >= 800

print(f'Starting generation of {TARGET} levels...')
print(f'  205 EASY   6x6  (2  constraints, ~16 clue cells)')
print(f'  205 MEDIUM 6x6  (3  constraints, ~13 clue cells)')
print(f'  205 HARD   8x8  (8  constraints, ~22 clue cells)')
print(f'  205 EXPERT 8x8  (16 constraints, ~17 clue cells)')
print()

levels = generate_batch(TARGET)

print(f'\n✅ {len(levels)} levels ready!')


In [ ]:
# ── Cell 8 · Full validation ───────────────────────────────────────────────────
# Re-checks every level independently of the generator.

def validate_level(lvl):
    size = lvl['size']
    sol  = lvl['solution']
    clues = lvl['clues']
    constraints = lvl['constraints']
    half = size // 2
    errors = []

    # 1. Solution must have equal FIRE/WATER per row and column
    for r in range(size):
        if sol[r].count(FIRE) != half or sol[r].count(WATER) != half:
            errors.append(f'row {r} balance')
    for c in range(size):
        col = [sol[r][c] for r in range(size)]
        if col.count(FIRE) != half or col.count(WATER) != half:
            errors.append(f'col {c} balance')

    # 2. No three consecutive
    for r in range(size):
        for c in range(size - 2):
            if sol[r][c] == sol[r][c+1] == sol[r][c+2]:
                errors.append(f'horiz triple at ({r},{c})')
    for c in range(size):
        for r in range(size - 2):
            if sol[r][c] == sol[r+1][c] == sol[r+2][c]:
                errors.append(f'vert triple at ({r},{c})')

    # 3. Clues are a valid subset of solution
    for r in range(size):
        for c in range(size):
            if clues[r][c] != EMPTY and clues[r][c] != sol[r][c]:
                errors.append(f'clue mismatch at ({r},{c})')

    # 4. Solution satisfies all constraints
    for con in constraints:
        e1 = sol[con['r1']][con['c1']]
        e2 = sol[con['r2']][con['c2']]
        if con['type'] == 'EQUAL' and e1 != e2:
            errors.append(f'constraint EQUAL violated at ({con["r1"]},{con["c1"]})')
        if con['type'] == 'DIFFERENT' and e1 == e2:
            errors.append(f'constraint DIFFERENT violated at ({con["r1"]},{con["c1"]})')

    # 5. Unique solution
    if count_solutions(clues, size, constraints, limit=2) != 1:
        errors.append('not uniquely solvable')

    return errors


print(f'Validating {len(levels)} levels...')
t0 = time.time()
fail_count = 0

for i, lvl in enumerate(levels):
    errs = validate_level(lvl)
    if errs:
        fail_count += 1
        print(f'  ❌ Level {lvl["levelNumber"]} ({lvl["difficulty"]} {lvl["size"]}x{lvl["size"]}): {errs}')
    if (i+1) % 100 == 0:
        print(f'  {i+1}/{len(levels)} checked...')

elapsed = time.time() - t0
print(f'\nValidation complete in {elapsed:.1f}s')
if fail_count == 0:
    print(f'✅ All {len(levels)} levels are VALID and UNIQUELY SOLVABLE!')
else:
    print(f'❌ {fail_count} levels failed validation')

# Stats
print('\n── Distribution ──────────────────────────')
diff_cnt = Counter(lvl['difficulty'] for lvl in levels)
size_cnt = Counter(lvl['size'] for lvl in levels)
for d in ['EASY', 'MEDIUM', 'HARD', 'EXPERT']:
    print(f'  {d:8s}: {diff_cnt.get(d, 0):4d} levels')
print(f'  6x6: {size_cnt.get(6,0)} | 8x8: {size_cnt.get(8,0)}')

In [ ]:
# ── Cell 9 · Export ────────────────────────────────────────────────────────────

# ── 9a: JSON with named values (FIRE / WATER / EMPTY) ─────────────────────────
def int_grid_to_names(grid):
    return [[CELL_NAMES[v] for v in row] for row in grid]

named_levels = [
    {**lvl,
     'clues':    int_grid_to_names(lvl['clues']),
     'solution': int_grid_to_names(lvl['solution'])}
    for lvl in levels
]
payload_named = {'version': 1, 'total': len(levels), 'levels': named_levels}
with open('solvex_levels.json', 'w') as f:
    json.dump(payload_named, f, separators=(',', ':'))
sz1 = len(json.dumps(payload_named)) // 1024
print(f'✅ solvex_levels.json  →  {sz1} KB  ({len(levels)} levels, named values)')

# ── 9b: Compact JSON with integer values 0/1/2 ────────────────────────────────
payload_compact = {'version': 1, 'total': len(levels), 'levels': levels}
with open('solvex_levels_compact.json', 'w') as f:
    json.dump(payload_compact, f, separators=(',', ':'))
sz2 = len(json.dumps(payload_compact)) // 1024
print(f'✅ solvex_levels_compact.json  →  {sz2} KB  (integer values 0/1/2)')

# ── 9c: Kotlin data file (drop into app/src/main/java/app/solvex/data/) ────────
def build_kotlin(levels):
    cell_str = {
        EMPTY: 'CellElement.EMPTY',
        FIRE:  'CellElement.FIRE',
        WATER: 'CellElement.WATER',
    }

    def row_kt(row):
        return 'listOf(' + ','.join(cell_str[v] for v in row) + ')'

    def grid_kt(grid):
        return 'listOf(' + ','.join(row_kt(r) for r in grid) + ')'

    def con_kt(c):
        return (f'Constraint({c["r1"]},{c["c1"]},{c["r2"]},{c["c2"]}'
                f',ConstraintType.{c["type"]})')

    lines = [
        'package app.solvex.data',
        '',
        'import app.solvex.model.*',
        '',
        '// Auto-generated by tools/solvex_level_generator.ipynb — DO NOT EDIT',
        f'// {len(levels)} levels · EASY/MEDIUM 6x6 · HARD/EXPERT 8x8',
        'object PrebuiltLevels {',
        '    val ALL: List<GameLevel> = listOf(',
    ]

    for i, lvl in enumerate(levels):
        comma = ',' if i < len(levels) - 1 else ''
        con_list = 'listOf(' + ','.join(con_kt(c) for c in lvl['constraints']) + ')'
        lines.append(
            f'        GameLevel('
            f'id={lvl["id"]},levelNumber={lvl["levelNumber"]},'
            f'size={lvl["size"]},difficulty=Difficulty.{lvl["difficulty"]},'
            f'clues={grid_kt(lvl["clues"])},'
            f'constraints={con_list},'
            f'solution={grid_kt(lvl["solution"])}){comma}'
        )

    lines.extend(['    )', '}'])
    return '\n'.join(lines)


kotlin_code = build_kotlin(levels)
with open('PrebuiltLevels.kt', 'w') as f:
    f.write(kotlin_code)
sz3 = len(kotlin_code) // 1024
print(f'✅ PrebuiltLevels.kt  →  {sz3} KB  (drop into app/solvex/data/)')

print('\n── Files ready ───────────────────────────────────────────')
print('  solvex_levels.json          — human-readable, load in Android Assets')
print('  solvex_levels_compact.json  — smaller file, same data')
print('  PrebuiltLevels.kt           — Kotlin object, no runtime parsing needed')

In [ ]:
# ── Cell 10 · Download files from Colab ───────────────────────────────────────
# Run this only if you are in Google Colab.

try:
    from google.colab import files
    print('Downloading files...')
    files.download('solvex_levels.json')
    files.download('solvex_levels_compact.json')
    files.download('PrebuiltLevels.kt')
    print('✅ Downloads started')
except ImportError:
    print('Not running in Colab — files are saved in the current directory.')
    import os
    for fn in ['solvex_levels.json', 'solvex_levels_compact.json', 'PrebuiltLevels.kt']:
        if os.path.exists(fn):
            print(f'  {fn}  ({os.path.getsize(fn)//1024} KB)')

## Using the output in the Android app

### Option A — Kotlin object (fastest, no I/O)

1. Copy `PrebuiltLevels.kt` to `app/src/main/java/app/solvex/data/`
2. In `GameViewModel.kt` change:
```kotlin
const val TOTAL_LEVELS = 820   // update count
```
3. Replace `PuzzleGenerator.generate(...)` call in `startLevel()` with:
```kotlin
val level = PrebuiltLevels.ALL
    .getOrNull(levelIndex)
    ?.copy(levelNumber = levelIndex + 1)
    ?: PuzzleGenerator.generate(GRID_SIZE, difficulty, seed)
        .copy(levelNumber = levelIndex + 1)
```

### Option B — JSON asset (smaller APK if many levels)

1. Copy `solvex_levels_compact.json` to `app/src/main/assets/`
2. Parse with Gson or kotlinx.serialization at app startup
3. Cache in memory or Room database

---
*Generated by solvex_level_generator.ipynb*